# 🎙️ E4 — Voix Off Qwen3-TTS (§7.2)

**Notebook modulaire** — chaque étape peut être lancée indépendamment selon le `MODE`.

## Modes disponibles

| MODE | Description | Durée estimée |
|------|-------------|---------------|
| `full` | Tout : upload voix → synthèse → qualité → state | ~5–10 min |
| `resume_after_fail` | Ignore upload voix, re-synthétise + qualité | ~2–3 min |
| `quality_check` | Vérifie seulement le WER sur un audio existant | ~30 sec |
| `voice_only` | Upload/denoising d'une nouvelle voix uniquement | ~1 min |

## ⚙️ Franco modifie uniquement la **Cell 1** (VIDEO_ID + MODE)

---

**Contrat E4 (§4.3) :**  
- Input : `03_script_tts.txt` (depuis A5) + échantillon voix + texte de référence  
- Output : `04_voixoff.wav`, `04_timestamps.json`, `04_rapport_audio.md`  
- State : `en_cours` → `termine` (WER ≤ 3%) ou `echec` (WER > 3%)

**Décisions architecturales résolues (§12) :**
- ✓ Modèle TTS : Qwen3-TTS (voice cloning)
- ✓ Débruitage : noisereduce v1
- ✓ Seuil WER : 3%
- ✓ Mise à jour state : uniquement si quality_pass
- ✓ Taux : 22050 Hz final
- ✓ Max tentatives WER : 3

---
## Cell 0 — Setup & Auth

**Toujours exécuter.** Installe les dépendances, monte Drive, configure les chemins.

> ⏱ Première exécution : ~3–5 min (téléchargement des packages). Les suivantes : ~30 sec (cache pip).

In [ ]:
# ── Cell 0 : Setup & Auth (TOUJOURS EXÉCUTER) ──────────────────────────────
import os
os.environ.setdefault('PYTHONHASHSEED', '0')

print("📦 Installation des dépendances...")
import subprocess

packages = [
    "qwen-tts",
    "faster-whisper",
    "noisereduce",
    "librosa",
    "soundfile",
    "pyloudnorm",
    "jiwer",  # WER calculation
]

for pkg in packages:
    result = subprocess.run(
        ["pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗"
    print(f"  {status} {pkg}")

print("\n🔗 Montage Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive monté")

# ── Imports globaux ──────────────────────────────────────────────────────────
import json
import time
import numpy as np
import soundfile as sf
import librosa
import noisereduce as nr
import pyloudnorm as pyln
from pathlib import Path
from datetime import datetime, timezone
import jiwer  # jiwer.wer(), jiwer.Compose(...) pour la normalisation WER (Cell 6)

print("✓ Imports OK")

# ── Fonctions utilitaires state.json ─────────────────────────────────────────
def _maintenant_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

def _lire_state(dossier_video):
    chemin = dossier_video / 'state.json'
    if not chemin.exists():
        raise FileNotFoundError(
            f"❌ state.json introuvable : {chemin}\n"
            f"   → Vérifie que VIDEO_ID est correct et que new-short a été lancé."
        )
    return json.loads(chemin.read_text(encoding='utf-8'))

def _ecrire_state(dossier_video, state):
    chemin = dossier_video / 'state.json'
    tmp = chemin.with_suffix('.tmp')
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, chemin)

def _historique(state, evenement, message):
    # L'agent vient du state, il n'est pas ecrit en dur : l'historique reel de
    # 2026-09-11_v01 contient "colab_voix_f5tts" ET "colab_voix_qwen_tts"
    # pour une meme etape, parce que la chaine etait recopiee a la main a
    # chaque changement de moteur. Deux noms pour un agent cassent tout
    # regroupement en aval (H1, corpus). Le moteur va dans le message.
    agent = (state.get('etapes', {}).get('E4_audio', {}).get('agent')) or 'colab_voix'
    state.setdefault('historique', []).append({
        'horodatage': _maintenant_iso(),
        'agent': agent,
        'evenement': evenement,
        'message': message,
    })

def etape_commencer(dossier_video):
    state = _lire_state(dossier_video)
    e = state['etapes']['E4_audio']
    statuts_ok = ('a_venir', 'attente_franco', 'echec', 'en_cours')
    if e['statut'] not in statuts_ok:
        raise RuntimeError(
            f"❌ E4_audio statut inattendu : '{e['statut']}' (attendu : {statuts_ok})\n"
            f"   → Si statut='termine', la vidéo est déjà produite."
        )

    # ── Plafond de tentatives (§2 : 3 essais puis alerte) ────────────────
    # La regle vivait uniquement dans l'Orchestrateur, qui ne tourne pas
    # entre deux runs Colab : c'est le notebook qu'on relance, pas lui. Sur
    # 2026-09-11_v01 le compteur est monte a 8 sans qu'aucune alerte ne
    # parte — 2 h 20 passees a relancer un moteur qui ne pouvait pas
    # marcher. La regle doit vivre la ou l'action a lieu.
    deja = e.get('tentatives', 0)
    plafond = globals().get('MAX_TENTATIVES', 3)
    forcer = globals().get('FORCER_RELANCE', False)
    if deja >= plafond and not forcer:
        dernier = e.get('message') or '(pas de message)'
        raise SystemExit(
            f"⛔ {deja} tentatives deja sur E4_audio (plafond : {plafond}).\n"
            f"   Dernier echec : {dernier}\n\n"
            f"   Relancer a l'identique ne donnera pas un autre resultat.\n"
            f"   → Lis 04_rapport_audio.md : il dit si l'echec est marginal\n"
            f"     (re-synthese utile) ou structurel (moteur ou reference en cause).\n"
            f"   → Pour passer outre en connaissance de cause :\n"
            f"     FORCER_RELANCE = True dans la Cell 1."
        )

    moteur = globals().get('MOTEUR_TTS', 'Qwen3-TTS')
    e['tentatives'] = deja + 1
    e['statut'] = 'en_cours'
    e['debut'] = _maintenant_iso()
    e['fin'] = None
    _historique(state, 'en_cours', f"Run Colab {moteur}, tentative {e['tentatives']}")
    _ecrire_state(dossier_video, state)
    print(f"✓ State mis à jour : E4_audio = en_cours (tentative {e['tentatives']}/{plafond})")

def etape_terminer(dossier_video, sorties, message):
    state = _lire_state(dossier_video)
    e = state['etapes']['E4_audio']
    e['statut'] = 'termine'
    e['fin'] = _maintenant_iso()
    e['sorties'] = sorties
    e['message'] = message
    _historique(state, 'termine', message)
    _ecrire_state(dossier_video, state)
    print(f"✅ State mis à jour : E4_audio = termine")

def etape_echouer(dossier_video, message):
    state = _lire_state(dossier_video)
    e = state['etapes']['E4_audio']
    e['statut'] = 'echec'
    e['fin'] = _maintenant_iso()
    e['message'] = message
    _historique(state, 'echec', message)
    _ecrire_state(dossier_video, state)
    print(f"⚠️ State mis à jour : E4_audio = echec")

print("\n✅ Cell 0 terminée — prêt pour Cell 1")

---
## Cell 1 — Config + MODE Selector

**⚠️ Seule cellule que Franco modifie.**

1. Renseigne `VIDEO_ID` (ex: `"2026-09-11_v01"`)
2. Choisis le `MODE` parmi : `full`, `resume_after_fail`, `quality_check`, `voice_only`
3. Lance toutes les cellules dans l'ordre (ou utilise *Runtime → Run all*)

```
MODE = "full"              → Premier run complet
MODE = "resume_after_fail" → WER > 3% au run précédent, re-synthétise
MODE = "quality_check"     → Vérification rapide WER d'un audio existant
MODE = "voice_only"        → Changer l'échantillon de voix uniquement
```

In [ ]:
# ── Cell 1 : Config + MODE ─────────────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FRANCO : laisse VIDEO_ID vide pour une detection automatique.  ║
# ║  Remplis-le seulement si plusieurs videos attendent l'audio,    ║
# ║  ou pour cibler une video precise.                               ║
# ╚══════════════════════════════════════════════════════════════════╝

VIDEO_ID = ""                   # ← vide = auto-detection. Sinon : "2026-09-11_v01"
MODE     = "voice_only"               # ← "full" | "resume_after_fail" | "quality_check" | "voice_only"

# ── Paramètres avancés (modifier si nécessaire) ──────────────────────────────
NOM_VOIX         = "voix_principale"  # nom du profil voix dans 00_Profil/voix/
DENOISE          = False              # debruitage noisereduce a l'upload — False si la
                                       # reference est deja propre (ex. voix ElevenLabs) :
                                       # sur un signal sans bruit reel, noisereduce n'a que
                                       # du signal a "corriger" et denature la voix.
GRAINE           = 42                 # seed Qwen3-TTS pour reproductibilité
MOTEUR_TTS       = "Qwen3-TTS"        # trace dans state.json/historique
SEUIL_WER        = 0.03               # 3% → au-dessus, l'audio est refusé (§12)
SEUIL_WER_GRAVE  = 0.30               # 30% → au-dessus, l'échec est structurel :
                                       # relancer la même config ne sert a rien.
                                       # Sur 2026-09-11_v01, 5 echecs
                                       # etaient a 93-98 % (fuite de reference
                                       # F5-TTS) ; les marginaux, a 8-18 %.
MAX_TENTATIVES   = 3                  # §2 : 3 essais par etape, puis alerte.
                                       # Applique par etape_commencer (Cell 0).
FORCER_RELANCE   = False              # True = passer outre le plafond, en
                                       # connaissance de cause
PAUSE_MS         = 250                # pause entre phrases (ms)
CIBLE_LUFS       = -16.0              # loudness cible YouTube (LUFS)
TAUX_FINAL_HZ    = 22050              # taux d'échantillonnage de sortie (Hz)

# ── Chemins Drive ─────────────────────────────────────────────────────────────
RACINE = Path('/content/drive/MyDrive/ChaineYouTube')  # ajuster si raccourci différent
if not RACINE.is_dir():
    raise SystemExit(f"❌ Racine Drive introuvable : {RACINE}\n   → Vérifie le chemin (raccourci Drive ?)")

# ── Auto-détection de VIDEO_ID si non renseigné ──────────────────────────────
# Statuts qui indiquent qu'un run Colab est attendu ou peut reprendre :
# attente_franco (prêt), echec (à reprendre), en_cours (run précédent coupé net).
STATUTS_CANDIDATS = ('attente_franco', 'echec', 'en_cours')

def _candidats_audio(racine):
    candidats = []
    dossier_videos = racine / 'videos'
    if not dossier_videos.is_dir():
        return candidats
    for dossier in sorted(dossier_videos.iterdir()):
        chemin_state = dossier / 'state.json'
        if not chemin_state.is_file():
            continue
        try:
            state = json.loads(chemin_state.read_text(encoding='utf-8-sig'))
        except (json.JSONDecodeError, OSError):
            continue
        statut = state.get('etapes', {}).get('E4_audio', {}).get('statut')
        if statut in STATUTS_CANDIDATS:
            candidats.append((state['video_id'], statut, state.get('titre_travail', '')))
    return candidats

if not VIDEO_ID:
    candidats = _candidats_audio(RACINE)
    if not candidats:
        raise SystemExit(
            "❌ Aucune vidéo n'attend l'audio en ce moment (E4_audio pas dans "
            f"{STATUTS_CANDIDATS}).\n   → Vérifie `/short-state` pour voir où en sont les vidéos."
        )
    if len(candidats) == 1:
        VIDEO_ID = candidats[0][0]
        print(f"🔎 Une seule vidéo en attente d'audio, sélection automatique : {VIDEO_ID}")
    else:
        lignes = "\n".join(f"   - {vid} ({statut}) — {titre}" for vid, statut, titre in candidats)
        raise SystemExit(
            f"❌ Plusieurs vidéos en attente d'audio, choix ambigu :\n{lignes}\n"
            f"   → Remplis VIDEO_ID ci-dessus avec l'une de ces valeurs, puis relance la Cell 1."
        )

DOSSIER_VIDEO  = RACINE / 'videos' / VIDEO_ID
DOSSIER_VOIX   = RACINE / '00_Profil' / 'voix' / NOM_VOIX

# ── Dérivation des flags conditionnels selon MODE ────────────────────────────
MODES_VALIDES = ('full', 'resume_after_fail', 'quality_check', 'voice_only')
if MODE not in MODES_VALIDES:
    raise ValueError(f"❌ MODE invalide : '{MODE}'. Choisir parmi {MODES_VALIDES}")

RUN_VOICE_UPLOAD  = MODE in ('full', 'voice_only')
RUN_SYNTHESIS     = MODE in ('full', 'resume_after_fail')
RUN_QUALITY       = MODE in ('full', 'resume_after_fail', 'quality_check')
RUN_STATE_UPDATE  = MODE in ('full', 'resume_after_fail', 'quality_check')

# ── Validation des prérequis ─────────────────────────────────────────────────
errors = []
if not DOSSIER_VIDEO.is_dir():
    errors.append(f"❌ Dossier vidéo introuvable : {DOSSIER_VIDEO}\n   → VIDEO_ID correct ?")
if RUN_SYNTHESIS and not (DOSSIER_VIDEO / '03_script_tts.txt').exists():
    errors.append(f"❌ 03_script_tts.txt manquant dans {DOSSIER_VIDEO}\n   → CP2 validé ? A5 Filtre TTS exécuté ?")
if MODE == 'quality_check' and not (DOSSIER_VIDEO / '04_voixoff.wav').exists():
    errors.append(f"❌ 04_voixoff.wav introuvable pour quality_check\n   → Lance d'abord MODE='full'")

if errors:
    print("\n".join(errors))
    raise SystemExit("Corrige les erreurs ci-dessus avant de continuer.")

# ── Résumé ───────────────────────────────────────────────────────────────────
print(f"""✅ Config OK
   VIDEO_ID  : {VIDEO_ID}
   MODE      : {MODE}
   Étapes actives :
     Upload voix    : {'✓ OUI' if RUN_VOICE_UPLOAD else '✗ ignorée'}
     Synthèse Qwen  : {'✓ OUI' if RUN_SYNTHESIS else '✗ ignorée'}
     Qualité WER    : {'✓ OUI' if RUN_QUALITY else '✗ ignorée'}
     Update state   : {'✓ OUI' if RUN_STATE_UPDATE else '✗ ignorée'}
""")


---
## Cell 2 — Upload voix & Débruitage

**Actif si** : `MODE = "full"` ou `MODE = "voice_only"`

- Premier run : upload `ref.wav` + `ref.txt` (phrase lue dans l'audio)
- Denoising automatique via `noisereduce`
- Voix stockée dans `00_Profil/voix/{NOM_VOIX}/v{N}/`

> 💡 **Format attendu :**
> - `ref.wav` : 3–10 secondes, voix claire, mono ou stéréo
> - `ref.txt` : phrase exacte prononcée dans le fichier audio (ex: `"Bonjour, je suis Franco et voici ma chaîne."`)  
> - Si une voix existe déjà, elle sera réutilisée sans upload.

In [ ]:
# ── Cell 2 : Upload voix & Débruitage ─────────────────────────────────────
ref_wav = None
ref_txt = None
v = None

if not RUN_VOICE_UPLOAD:
    print(f"⏭️ Cell 2 ignorée (MODE='{MODE}')")
else:
    def version_courante(dossier_voix):
        if not dossier_voix.is_dir():
            return 0
        versions = [
            int(p.name[1:]) for p in dossier_voix.iterdir()
            if p.name.startswith('v') and p.name[1:].isdigit()
        ]
        return max(versions, default=0)

    # Le picker Colab n'impose pas le format : on accepte les formats audio
    # courants et on convertit vers wav via ffmpeg si besoin, plutot que de
    # rejeter silencieusement un .mp3/.m4a (ce qui laissait `wavs` vide et
    # faisait croire qu'aucun fichier n'avait ete uploade).
    import subprocess
    import tempfile as _tempfile

    EXTENSIONS_AUDIO = ('.wav', '.mp3', '.m4a', '.ogg', '.flac', '.aac')

    def _est_audio(nom):
        return nom.lower().endswith(EXTENSIONS_AUDIO)

    def _charger_audio_uploade(nom, contenu):
        suffixe = Path(nom).suffix.lower() or '.bin'
        with _tempfile.NamedTemporaryFile(suffix=suffixe, delete=False) as f_src:
            f_src.write(contenu)
            chemin_src = f_src.name
        if suffixe == '.wav':
            chemin_a_lire = chemin_src
        else:
            chemin_a_lire = chemin_src + '.wav'
            resultat = subprocess.run(
                ['ffmpeg', '-y', '-i', chemin_src, '-ar', '44100', '-ac', '1', chemin_a_lire],
                capture_output=True, text=True,
            )
            if resultat.returncode != 0:
                raise ValueError(f"❌ Conversion {suffixe} → wav impossible : {resultat.stderr[-300:]}")
            print(f"   🔀 {nom} converti {suffixe} → wav (ffmpeg)")
        return sf.read(chemin_a_lire)

    voix_existante = version_courante(DOSSIER_VOIX)

    if voix_existante == 0:
        # ── Pas de voix stockée : upload obligatoire ──────────────────────────
        print("🎤 Aucune voix stockée. Upload requis...")
        print("   → Sélectionne 2 fichiers : ref.wav (audio) + ref.txt (texte lu)")
        from google.colab import files
        uploades = files.upload()

        wavs = [n for n in uploades if _est_audio(n)]
        txts = [n for n in uploades if n.lower().endswith('.txt')]

        if len(wavs) != 1:
            raise ValueError(f"❌ Exactement 1 fichier audio attendu ({EXTENSIONS_AUDIO}), reçu : {wavs}")
        if len(txts) != 1:
            raise ValueError(f"❌ Exactement 1 .txt attendu, reçu : {txts}")

        # Lecture (+ conversion si besoin) + débruitage noisereduce optionnel
        audio_raw, taux_src = _charger_audio_uploade(wavs[0], uploades[wavs[0]])
        if audio_raw.ndim > 1:
            audio_raw = audio_raw.mean(axis=1)  # stéréo → mono
        audio_raw = audio_raw.astype(np.float32)

        if DENOISE:
            print("🔇 Débruitage noisereduce...")
            audio_denoise = nr.reduce_noise(y=audio_raw, sr=taux_src, stationary=False)
        else:
            print("⏭️  Débruitage ignoré (DENOISE=False)")
            audio_denoise = audio_raw
        print(f"   Durée : {len(audio_denoise)/taux_src:.2f}s | Taux : {taux_src} Hz")

        # Sauvegarde versionnée
        v = voix_existante + 1
        dossier_v = DOSSIER_VOIX / f'v{v}'
        dossier_v.mkdir(parents=True, exist_ok=True)

        ref_wav = dossier_v / 'ref.wav'
        sf.write(ref_wav, audio_denoise, taux_src)

        ref_txt = uploades[txts[0]].decode('utf-8').strip()
        (dossier_v / 'ref.txt').write_text(ref_txt, encoding='utf-8')
        (dossier_v / 'meta.json').write_text(json.dumps({
            'cree_le': _maintenant_iso(),
            'source_wav': wavs[0],
            'source_txt': txts[0],
            'taux_hz': taux_src,
            'denoised': DENOISE,
        }, ensure_ascii=False, indent=2), encoding='utf-8')

        print(f"✅ Voix stockée : {NOM_VOIX} v{v} ({dossier_v})")

    else:
        # ── Voix existante : proposer mise à jour ou réutiliser ───────────────
        dossier_v = DOSSIER_VOIX / f'v{voix_existante}'
        ref_wav = dossier_v / 'ref.wav'
        ref_txt = (dossier_v / 'ref.txt').read_text(encoding='utf-8')
        v = voix_existante

        if MODE == 'voice_only':
            # En mode voice_only avec voix existante : upload nouvelle version
            print(f"🔄 voice_only : upload d'une nouvelle version (v{v} actuelle)...")
            from google.colab import files
            uploades = files.upload()
            wavs = [n for n in uploades if _est_audio(n)]
            txts = [n for n in uploades if n.lower().endswith('.txt')]

            if wavs and txts:
                audio_raw, taux_src = _charger_audio_uploade(wavs[0], uploades[wavs[0]])
                if audio_raw.ndim > 1:
                    audio_raw = audio_raw.mean(axis=1)
                audio_raw = audio_raw.astype(np.float32)
                if DENOISE:
                    print("🔇 Débruitage noisereduce...")
                    audio_denoise = nr.reduce_noise(y=audio_raw, sr=taux_src, stationary=False)
                else:
                    print("⏭️  Débruitage ignoré (DENOISE=False)")
                    audio_denoise = audio_raw

                v = voix_existante + 1
                dossier_v = DOSSIER_VOIX / f'v{v}'
                dossier_v.mkdir(parents=True, exist_ok=True)
                ref_wav = dossier_v / 'ref.wav'
                sf.write(ref_wav, audio_denoise, taux_src)
                ref_txt = uploades[txts[0]].decode('utf-8').strip()
                (dossier_v / 'ref.txt').write_text(ref_txt, encoding='utf-8')
                (dossier_v / 'meta.json').write_text(json.dumps({
                    'cree_le': _maintenant_iso(), 'source_wav': wavs[0],
                    'taux_hz': taux_src, 'denoised': DENOISE,
                }, ensure_ascii=False, indent=2), encoding='utf-8')
                print(f"✅ Nouvelle voix : {NOM_VOIX} v{v}")
            else:
                print(f"ℹ️ Aucun fichier uploadé — voix v{v} conservée")
        else:
            print(f"✅ Voix chargée : {NOM_VOIX} v{v} ({dossier_v})")

    print(f"   ref.txt : {repr(ref_txt[:80])}{'...' if ref_txt and len(ref_txt) > 80 else ''}")

---
## Cell 3 — Chargement Qwen3-TTS + faster-whisper

**Toujours exécuter** (les modèles sont mis en cache après le premier chargement).

- Qwen3-TTS : modèle de clonage vocal
- faster-whisper `large-v3-turbo` : transcription pour calcul WER
- GPU T4 recommandé (⚡ plus rapide) — fallback CPU si GPU indisponible

> ⏱ Premier chargement : ~2 min (téléchargement poids). Cache Colab : ~10 sec.

In [ ]:
# ── Cell 3 : Chargement modèles (TOUJOURS EXÉCUTER) ───────────────────────
import torch
from faster_whisper import WhisperModel

# ── Détection GPU ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"⚡ GPU détecté : {gpu_name} ({gpu_mem:.1f} GB)")
else:
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"
    print("⚠️  Pas de GPU — mode CPU (lent, ~3-5x plus long)")
    print("   → Pour activer GPU : Runtime → Change runtime type → T4 GPU")

# ── Chargement Qwen3-TTS ───────────────────────────────────────────────────
# Remplace F5-TTS (Session 6) : sur plusieurs echantillons de reference
# differents (enregistrement reel, voix ElevenLabs), F5-TTS "fuyait"
# periodiquement le contenu de sa reference au lieu du texte cible — signe
# d'un probleme structurel plutot que d'un mauvais echantillon. Qwen3-TTS
# clone avec 3-10s de reference (contre 10-15s+ vise difficilement avec
# F5-TTS) et une API differente (generate_voice_clone).
_qwen_tts_model = None

def get_qwen_tts():
    global _qwen_tts_model
    if _qwen_tts_model is None:
        print("📥 Chargement Qwen3-TTS...")
        from qwen_tts import Qwen3TTSModel
        try:
            _qwen_tts_model = Qwen3TTSModel.from_pretrained(
                "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
                device_map="cuda:0" if DEVICE == "cuda" else "cpu",
                dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            )
            print("✅ Qwen3-TTS chargé")
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("⚠️  OOM GPU — tentative sur CPU...")
                _qwen_tts_model = Qwen3TTSModel.from_pretrained(
                    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
                    device_map="cpu",
                    dtype=torch.float32,
                )
                print("✅ Qwen3-TTS chargé (CPU — plus lent)")
            else:
                raise
    return _qwen_tts_model

# ── Chargement Whisper ─────────────────────────────────────────────────────
_whisper_model = None

def get_whisper():
    global _whisper_model
    if _whisper_model is None:
        print("📥 Chargement faster-whisper large-v3-turbo...")
        try:
            _whisper_model = WhisperModel(
                "large-v3-turbo", device=DEVICE, compute_type=COMPUTE_TYPE
            )
        except Exception as e:
            print(f"⚠️  Erreur {DEVICE} : {e}\n   Fallback CPU...")
            _whisper_model = WhisperModel(
                "large-v3-turbo", device="cpu", compute_type="int8"
            )
        print("✅ Whisper chargé")
    return _whisper_model

# Pré-chargement si on va en avoir besoin
if RUN_SYNTHESIS:
    get_qwen_tts()
if RUN_QUALITY:
    get_whisper()

print("\n✅ Cell 3 terminée")

---
## Cell 4 — Synthèse audio (Qwen3-TTS)

**Actif si** : `MODE = "full"` ou `MODE = "resume_after_fail"`

- Lit `03_script_tts.txt` (une phrase par ligne)
- Synthétise chaque phrase via Qwen3-TTS avec clonage de voix
- Graine fixe `GRAINE` pour reproductibilité

> ⏱ ~2–4 secondes par phrase sur GPU T4.

In [ ]:
# ── Cell 4 : Synthèse audio ────────────────────────────────────────────────
clips_bruts = []      # liste de (audio_np, taux) — une entrée par phrase
phrases = []

if not RUN_SYNTHESIS:
    print(f"⏭️ Cell 4 ignorée (MODE='{MODE}')")
    # En quality_check, on charge le wav existant plus tard
else:
    # ── Vérification prérequis voix ───────────────────────────────────────────
    if ref_wav is None or ref_txt is None:
        # Charger la voix existante si Cell 2 a été ignorée
        def version_courante(dossier_voix):
            if not dossier_voix.is_dir():
                return 0
            versions = [
                int(p.name[1:]) for p in dossier_voix.iterdir()
                if p.name.startswith('v') and p.name[1:].isdigit()
            ]
            return max(versions, default=0)

        v = version_courante(DOSSIER_VOIX)
        if v == 0:
            raise FileNotFoundError(
                f"❌ Aucune voix stockée dans {DOSSIER_VOIX}\n"
                f"   → Lance d'abord MODE='full' ou MODE='voice_only'"
            )
        dossier_v = DOSSIER_VOIX / f'v{v}'
        ref_wav = dossier_v / 'ref.wav'
        ref_txt = (dossier_v / 'ref.txt').read_text(encoding='utf-8')
        print(f"✅ Voix chargée : {NOM_VOIX} v{v}")

    # ── Lecture script ────────────────────────────────────────────────────────
    script_path = DOSSIER_VIDEO / '03_script_tts.txt'
    phrases = [
        l.strip() for l in script_path.read_text(encoding='utf-8').splitlines()
        if l.strip()
    ]
    print(f"📝 Script : {len(phrases)} phrases à synthétiser")
    for i, p in enumerate(phrases):
        print(f"   [{i+1}] {p[:80]}{'...' if len(p) > 80 else ''}")

    # ── Marquage state en_cours ───────────────────────────────────────────────
    etape_commencer(DOSSIER_VIDEO)

    # ── Synthèse phrase par phrase ────────────────────────────────────────────
    qwen = get_qwen_tts()

    def synthetiser_phrase(phrase, graine):
        torch.manual_seed(graine)
        try:
            wavs, sr = qwen.generate_voice_clone(
                text=phrase,
                language="English",
                ref_audio=str(ref_wav),
                ref_text=ref_txt,
            )
            return wavs[0].astype(np.float32), int(sr)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"⚠️  OOM sur phrase {i+1} — vidage cache GPU...")
                torch.cuda.empty_cache()
                time.sleep(2)
                wavs, sr = qwen.generate_voice_clone(
                    text=phrase, language="English",
                    ref_audio=str(ref_wav), ref_text=ref_txt,
                )
                return wavs[0].astype(np.float32), int(sr)
            raise

    print("\n🎙️ Synthèse en cours...")
    t0 = time.time()
    for i, phrase in enumerate(phrases):
        print(f"   [{i+1}/{len(phrases)}] ", end="", flush=True)
        audio_np, taux = synthetiser_phrase(phrase, GRAINE + i)
        clips_bruts.append((audio_np, taux))
        duree = len(audio_np) / taux
        print(f"{duree:.1f}s ✓")

    total = time.time() - t0
    print(f"\n✅ Synthèse terminée — {total:.0f}s total ({len(phrases)} phrases)")

---
## Cell 5 — Extraction timestamps mot-à-mot

**Actif si** : `MODE = "full"` ou `MODE = "resume_after_fail"` ou `MODE = "quality_check"`

- Assemble les clips, normalise le volume (LUFS)
- Sauvegarde `04_voixoff.wav`
- Extrait les timestamps mot-à-mot via faster-whisper → `04_timestamps.json`

> 💡 Ces timestamps sont requis par E6 Montage pour la synchronisation sous-titres/images.

In [ ]:
# ── Cell 5 : Assemblage + Timestamps ──────────────────────────────────────
chemin_wav = DOSSIER_VIDEO / '04_voixoff.wav'
mots_timestamps = []

if not RUN_QUALITY:
    print(f"⏭️ Cell 5 ignorée (MODE='{MODE}')")
else:
    whisper = get_whisper()

    if RUN_SYNTHESIS and clips_bruts:
        # ── Assemblage depuis les clips bruts synthétisés ─────────────────────
        print("🔧 Assemblage des clips...")
        taux_final = TAUX_FINAL_HZ
        silence = np.zeros(int(taux_final * PAUSE_MS / 1000), dtype=np.float32)

        # Bornes de chaque phrase dans l'audio assemble. C'est la seule
        # etape du pipeline qui les connaisse exactement : apres coup, on ne
        # peut que les deviner en realignant les mots transcrits sur le
        # script, ce que le WER non nul rend fragile. A6 fait une scene par
        # phrase (§8), donc ces bornes permettent a A7 de recaler les durees
        # de scenes sur la voix off au lieu de garder l'estimation a
        # ~2.5 mots/s du storyboard.
        morceaux = []
        bornes_phrases = []
        curseur = 0
        for i, (audio_np, taux) in enumerate(clips_bruts):
            if taux != taux_final:
                audio_np = librosa.resample(audio_np, orig_sr=taux, target_sr=taux_final)
            if i > 0:
                morceaux.append(silence.copy())
                curseur += len(silence)
            debut_echantillon = curseur
            morceaux.append(audio_np)
            curseur += len(audio_np)
            bornes_phrases.append({
                'index': i + 1,
                'texte': phrases[i] if i < len(phrases) else '',
                'debut_s': round(debut_echantillon / taux_final, 3),
                'fin_s': round(curseur / taux_final, 3),
            })

        audio_assemble = np.concatenate(morceaux)

        # Normalisation LUFS
        meter = pyln.Meter(taux_final)
        loudness = meter.integrated_loudness(audio_assemble)
        if np.isfinite(loudness):
            audio_final = pyln.normalize.loudness(audio_assemble, loudness, CIBLE_LUFS)
            print(f"   Volume : {loudness:.1f} LUFS → {CIBLE_LUFS} LUFS")
        else:
            audio_final = audio_assemble
            print("⚠️  Normalisation LUFS ignorée (signal trop court/silencieux)")

        sf.write(chemin_wav, audio_final, taux_final)
        print(f"✅ {chemin_wav.name} sauvegardé ({len(audio_final)/taux_final:.1f}s, {taux_final} Hz)")

        # La normalisation LUFS change le gain, jamais la duree : les bornes
        # calculees avant restent valables sur le fichier ecrit.
        (DOSSIER_VIDEO / '04_phrases.json').write_text(
            json.dumps({
                'phrases': bornes_phrases,
                'duree_totale_s': round(len(audio_final) / taux_final, 3),
                'pause_ms': PAUSE_MS,
                'taux_hz': taux_final,
            }, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        print(f"✅ 04_phrases.json sauvegardé ({len(bornes_phrases)} phrases)")

    elif chemin_wav.exists():
        # ── Chargement d'un wav existant (quality_check) ──────────────────────
        print(f"📂 Chargement audio existant : {chemin_wav.name}")
        audio_final, taux_final = sf.read(str(chemin_wav), dtype='float32')
        if audio_final.ndim > 1:
            audio_final = audio_final.mean(axis=1)
        print(f"   Durée : {len(audio_final)/taux_final:.1f}s | Taux : {taux_final} Hz")

        # Charger aussi les phrases pour le WER
        script_path = DOSSIER_VIDEO / '03_script_tts.txt'
        if script_path.exists():
            phrases = [
                l.strip() for l in script_path.read_text(encoding='utf-8').splitlines()
                if l.strip()
            ]
    else:
        raise FileNotFoundError(
            f"❌ Aucun audio disponible (ni clips synthétisés, ni {chemin_wav.name})\n"
            f"   → Lance d'abord MODE='full'"
        )

    # ── Extraction timestamps mot-à-mot ───────────────────────────────────────
    print("⏱️  Extraction timestamps mot-à-mot...")
    segments, _ = whisper.transcribe(
        str(chemin_wav),
        word_timestamps=True,
        language="en",     # la chaine est entierement en anglais (doc archi §1)
        vad_filter=True,   # suppression silences VAD
    )

    mots_timestamps = [
        {'mot': w.word.strip(), 'debut_s': round(w.start, 3), 'fin_s': round(w.end, 3)}
        for seg in segments
        for w in (seg.words or [])
    ]

    texte_transcrit = ' '.join(m['mot'] for m in mots_timestamps)

    (DOSSIER_VIDEO / '04_timestamps.json').write_text(
        json.dumps(mots_timestamps, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f"✅ {len(mots_timestamps)} mots horodatés → 04_timestamps.json")
    print(f"   Transcription : {texte_transcrit[:120]}{'...' if len(texte_transcrit) > 120 else ''}")

---
## Cell 6 — Contrôle qualité WER

**Actif si** : `MODE = "full"`, `"resume_after_fail"` ou `"quality_check"`

- Compare le texte du script vs la transcription Whisper
- Calcule le **Word Error Rate (WER)** global
- Seuil : **3%** (si WER > 3% → `echec`, relancer avec `resume_after_fail`)

> 📊 WER = (insertions + suppressions + substitutions) / nombre de mots de référence

In [ ]:
# ── Cell 6 : Contrôle qualité WER ─────────────────────────────────────────
quality_pass = False
wer_global = None
texte_ref_global = ""
texte_transcrit_global = ""

if not RUN_QUALITY:
    print(f"⏭️ Cell 6 ignorée (MODE='{MODE}')")
else:
    # ── Texte de référence ────────────────────────────────────────────────────
    if phrases:
        texte_ref_global = ' '.join(phrases)
    else:
        # Fallback : lire le script si phrases non chargées
        script_path = DOSSIER_VIDEO / '03_script_tts.txt'
        if script_path.exists():
            phrases = [l.strip() for l in script_path.read_text(encoding='utf-8').splitlines() if l.strip()]
            texte_ref_global = ' '.join(phrases)
        else:
            raise FileNotFoundError(f"❌ 03_script_tts.txt introuvable — impossible de calculer le WER")

    # ── Texte transcrit ───────────────────────────────────────────────────────
    texte_transcrit_global = ' '.join(m['mot'] for m in mots_timestamps) if mots_timestamps else ""

    if not texte_transcrit_global:
        raise RuntimeError(
            "❌ Pas de transcription disponible pour WER.\n"
            "   → Cell 5 a-t-elle été exécutée ?"
        )

    # ── Calcul WER ────────────────────────────────────────────────────────────
    # Sans normalisation, jiwer ne retire que les espaces multiples (voir
    # jiwer.wer_default) : la ponctuation et la casse comptent comme des
    # erreurs de mots alors qu'elles ne se "prononcent" pas. Sur un texte
    # identique mais avec une ponctuation differente (script vs sortie
    # Whisper), ca peut a lui seul gonfler le WER de plusieurs dizaines de
    # points — observe en conditions reelles (9.21% de WER dont la moitie
    # des ecarts etaient juste virgule/point/majuscule).
    normalisation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords(),
    ])
    wer_global = jiwer.wer(
        reference=texte_ref_global,
        hypothesis=texte_transcrit_global,
        reference_transform=normalisation,
        hypothesis_transform=normalisation,
    )
    quality_pass = wer_global <= SEUIL_WER

    # ── Rapport console ───────────────────────────────────────────────────────
    print(f"""📊 Résultats WER
   WER global     : {wer_global:.2%}  (seuil : {SEUIL_WER:.0%})
   Verdict        : {'✅ PASS' if quality_pass else '❌ FAIL — WER > 3%'}
   Mots référence : {len(texte_ref_global.split())}
   Mots transcrits: {len(texte_transcrit_global.split())}
""")

    # ── Diagnostic gradue ─────────────────────────────────────────────────
    # Un seuil unique donnait les memes conseils a 96 % qu'a 9 % de WER. Sur
    # 2026-09-11_v01, les quatre premiers runs etaient a 93-98 % — la fuite
    # de reference de F5-TTS, un defaut structurel — et le notebook a
    # repondu quatre fois "re-synthese avec une autre graine". C'est ce
    # conseil suivi quatre fois qui a coute 2 h 20.
    echec_structurel = wer_global is not None and wer_global > SEUIL_WER_GRAVE

    if quality_pass:
        print(f"✅ Audio validé — WER {wer_global:.2%} ≤ {SEUIL_WER:.0%}")
    elif echec_structurel:
        print(f"""🛑 WER {wer_global:.2%} — echec STRUCTUREL (> {SEUIL_WER_GRAVE:.0%}).

   A ce niveau, ce n'est pas un alea de synthese : le texte genere n'a
   presque rien a voir avec le script. Relancer avec une autre graine ne
   changera rien — ne le fais pas.

   Causes probables, dans l'ordre :
   1. Le MOTEUR fuit le contenu de la reference dans la sortie
      (defaut observe sur F5-TTS, reproduit sur deux echantillons).
   2. L'echantillon de reference est inutilisable (mauvaise langue,
      trop court, plusieurs voix, musique de fond).
   3. La transcription de reference (ref.txt) ne correspond pas au ref.wav.

   → Verifie le diff reference/transcrit dans 04_rapport_audio.md : s'il
     ressemble au contenu de ta voix de reference, c'est la cause 1.
""")
    else:
        print(f"""⚠️  WER {wer_global:.2%} — echec marginal (seuil {SEUIL_WER:.0%}).

   L'audio ressemble au script mais reste au-dessus du seuil. Une
   re-synthese a de bonnes chances de passer :
   1. MODE = 'resume_after_fail' + GRAINE = {GRAINE + 1}
   2. Si ca echoue encore : regarde quelles phrases sont fautives dans le
      diff, elles sont souvent trop longues ou pleines de sigles.
   3. MODE = 'voice_only' pour un meilleur echantillon de voix.

   Plafond : {MAX_TENTATIVES} tentatives (§2). Au-dela, le notebook
   refusera de demarrer sans FORCER_RELANCE = True.
""")

---
## Cell 7 — Rapport + Mise à jour state.json

**Actif si** : `MODE = "full"`, `"resume_after_fail"` ou `"quality_check"`

- Génère `04_rapport_audio.md`
- Met à jour `state.json` : `E4_audio = "termine"` si WER ≤ 3%, `"echec"` sinon

> ⚠️ La mise à jour state n'est effectuée **que si** la qualité est vérifiée (sécurité contre les faux positifs).

In [ ]:
# ── Cell 7 : Rapport + State ───────────────────────────────────────────────
if not RUN_STATE_UPDATE:
    print(f"⏭️ Cell 7 ignorée (MODE='{MODE}')")
else:
    # ── Rapport Markdown ──────────────────────────────────────────────────────
    now_str = _maintenant_iso()
    duree_s = 0.0
    if chemin_wav.exists():
        audio_info, sr_info = sf.read(str(chemin_wav), dtype='float32')
        duree_s = len(audio_info) / sr_info if audio_info.ndim == 1 else len(audio_info[:, 0]) / sr_info

    rapport_lignes = [
        "# Rapport Audio E4",
        "",
        f"**Vidéo** : `{VIDEO_ID}`",
        f"**Date** : {now_str}",
        f"**Modèle TTS** : Qwen3-TTS",
        f"**Voix** : {NOM_VOIX} v{v if v else '?'}",
        f"**Graine** : {GRAINE}",
        "",
        "## Métriques qualité",
        "",
        f"| Métrique | Valeur | Seuil | Verdict |",
        f"|----------|--------|-------|---------|",
        f"| WER global | {f'{wer_global:.2%}' if wer_global is not None else 'N/A'} | {SEUIL_WER:.0%} | {'✅ PASS' if quality_pass else '❌ FAIL'} |",
        f"| Durée audio | {duree_s:.1f}s | — | — |",
        f"| Phrases | {len(phrases)} | — | — |",
        f"| Mots horodatés | {len(mots_timestamps)} | — | — |",
        "",
        "## Fichiers produits",
        "",
        f"- `04_voixoff.wav` — Audio final ({TAUX_FINAL_HZ} Hz, "
        f"{'débruité, ' if DENOISE else ''}normalisé {CIBLE_LUFS} LUFS)",
        "- `04_timestamps.json` — Timestamps mot-à-mot (pour les sous-titres, E6)",
        "- `04_phrases.json` — Bornes début/fin par phrase (recalage des scènes, E6)",
        "- `04_rapport_audio.md` — Ce rapport",
        "",
    ]

    if not quality_pass and wer_global is not None:
        structurel = wer_global > SEUIL_WER_GRAVE
        rapport_lignes += [
            "## ⚠️ Diagnostic WER",
            "",
            f"WER mesuré : **{wer_global:.2%}** (seuil : {SEUIL_WER:.0%}, "
            f"seuil structurel : {SEUIL_WER_GRAVE:.0%})",
            f"**Nature de l'échec : {'STRUCTUREL' if structurel else 'marginal'}**",
            "",
        ]
        if structurel:
            rapport_lignes += [
                "Le texte généré n'a presque rien à voir avec le script. Ce n'est",
                "pas un aléa de synthèse : **relancer avec une autre graine ne",
                "changera rien.**",
                "",
                "**Causes probables :**",
                "1. Le moteur fuit le contenu de la référence dans la sortie",
                "   (défaut observé sur F5-TTS, reproduit sur deux échantillons).",
                "2. L'échantillon de référence est inutilisable (langue, durée,",
                "   plusieurs voix, musique de fond).",
                "3. `ref.txt` ne correspond pas à `ref.wav`.",
                "",
                "Si le diff ci-dessous ressemble au contenu de la voix de",
                "référence plutôt qu'au script, c'est la cause 1.",
                "",
            ]
        else:
            rapport_lignes += [
                "L'audio ressemble au script mais reste au-dessus du seuil.",
                "",
                "**Actions recommandées :**",
                f"1. Re-synthèse : `MODE = 'resume_after_fail'` + `GRAINE = {GRAINE + 1}`",
                "2. Regarder quelles phrases sont fautives dans le diff : souvent",
                "   trop longues, ou chargées en sigles.",
                "3. Nouvelle voix : `MODE = 'voice_only'`",
                "",
            ]
        rapport_lignes += [
            "**Diff référence vs transcrit :**",
            "",
            f"*Référence* : `{texte_ref_global[:200]}{'...' if len(texte_ref_global) > 200 else ''}`",
            f"*Transcrit* : `{texte_transcrit_global[:200]}{'...' if len(texte_transcrit_global) > 200 else ''}`",
        ]

    (DOSSIER_VIDEO / '04_rapport_audio.md').write_text(
        '\n'.join(rapport_lignes), encoding='utf-8'
    )
    print("✅ 04_rapport_audio.md généré")

    # ── Mise à jour state.json ────────────────────────────────────────────────
    sorties = ['04_voixoff.wav', '04_timestamps.json', '04_rapport_audio.md']
    if (DOSSIER_VIDEO / '04_phrases.json').exists():
        # Consomme par A7 (construire_props.py --phrases) pour recaler les
        # durees de scenes sur l'audio reel. Absent en MODE='quality_check'
        # si le run d'origine est anterieur a la revue du 11/09/2026.
        sorties.insert(2, '04_phrases.json')

    if quality_pass:
        etape_terminer(
            DOSSIER_VIDEO,
            sorties,
            f"Audio validé — WER {wer_global:.2%}, {len(phrases)} phrases, {duree_s:.1f}s"
        )
    else:
        msg_wer = f"WER {wer_global:.2%} > seuil {SEUIL_WER:.0%}" if wer_global is not None else "WER non calculé"
        etape_echouer(DOSSIER_VIDEO, f"{msg_wer} — voir 04_rapport_audio.md")

---
## Cell 8 — Résumé & Prochaines étapes

**Toujours exécuter** — affiche le bilan et les actions à prendre.

In [ ]:
# ── Cell 8 : Résumé & Prochaines étapes ───────────────────────────────────
print("=" * 60)
print(f"  BILAN E4 — {VIDEO_ID}")
print("=" * 60)

# État state.json actuel
try:
    state_final = _lire_state(DOSSIER_VIDEO)
    e4 = state_final['etapes']['E4_audio']
    statut_e4 = e4['statut']
    tentatives_e4 = e4.get('tentatives', 0)
except Exception as ex:
    statut_e4 = f"(erreur lecture state : {ex})"
    tentatives_e4 = '?'

# Fichiers produits
fichiers = ['04_voixoff.wav', '04_timestamps.json', '04_rapport_audio.md']
print("\n📁 Fichiers :")
for f in fichiers:
    chemin_f = DOSSIER_VIDEO / f
    if chemin_f.exists():
        taille = chemin_f.stat().st_size
        print(f"   ✅ {f} ({taille/1024:.1f} KB)")
    else:
        print(f"   ✗  {f} (absent)")

print(f"\n📊 State : E4_audio = {statut_e4} (tentative {tentatives_e4})")

if wer_global is not None:
    print(f"   WER : {wer_global:.2%} {'✅' if quality_pass else '❌'}")

print()

# Instructions selon résultat
# MODE == 'voice_only' teste en premier : ce mode ne touche jamais state.json
# (RUN_STATE_UPDATE=False), donc statut_e4 reflète encore le run precedent —
# sans cette priorite, un upload de voix reussi affichait le vieil "echec".
if MODE == 'voice_only':
    # Cell 2 laisse toujours `v` (version finale) et `voix_existante` (version
    # avant ce run) en variables globales : v > voix_existante seulement si un
    # nouveau fichier a vraiment ete uploade. Sans ce test, ce bloc annoncait
    # "voix mise a jour" meme quand l'upload etait vide/annule, et la
    # resynthese suivante repartait sur l'ancienne reference en silence.
    upload_reussi = v is not None and 'voix_existante' in globals() and v > voix_existante
    if upload_reussi:
        print(f"""🎤 Voix mise à jour : v{v}.

   → Lance maintenant MODE = 'full' ou 'resume_after_fail' pour synthétiser.
""")
    else:
        print(f"""⚠️  Aucun nouveau fichier n'a été uploadé — voix v{v if v else '?'} inchangée.

   → Relance la Cell 2 et vérifie que tu sélectionnes bien UN .wav ET UN .txt
     dans le sélecteur de fichiers Colab (les deux sont obligatoires).
""")
elif statut_e4 == 'termine':
    print("""✅ E4 TERMINÉ avec succès !

   → L'Orchestrateur peut maintenant avancer à E5 (Storyboard).
   → Dans Claude : dis "lance l'orchestrateur" ou attend le prochain run auto.
   → E5 (A6 Designer) génère le storyboard visuel.
   → E6 (A7 Monteur + Remotion) rend le MP4 final.
""")
elif statut_e4 == 'echec':
    print(f"""❌ E4 EN ÉCHEC (WER trop élevé)

   Options selon la cause :

   🔄 Re-synthèse (même voix, autre graine) :
      MODE     = 'resume_after_fail'
      GRAINE   = {GRAINE + 1}   ← essaie +1, +2, +3...

   🎤 Nouvelle voix (enregistrement de meilleure qualité) :
      MODE = 'voice_only'  → upload nouveau ref.wav
      puis MODE = 'full'   → re-synthèse complète

   📝 Revoir le script (phrases trop longues ?) :
      → Demande à A5 de re-découper (CP2 → refus → A5 rerun)

   ⚠️  Max 3 tentatives total (§12) — après : signaler à Franco
""")
else:
    print(f"   Statut : {statut_e4} — vérifier les cellules précédentes.")

print("=" * 60)